# Aula 10 - Checkpoint: Contexto Longo

## Sua missao
Implemente `reduzir_contexto(documentos, pergunta, orcamento)`. A funcao deve reduzir o dossie do **Projeto Aurora**, preservar os cinco fatos necessarios e permitir que um LLM responda corretamente.

Esta e a unica atividade obrigatoria do checkpoint.

## Fluxo obrigatorio
1. Execute o cenario Aurora e observe a falha por excesso de contexto.
2. Implemente `reduzir_contexto`.
3. Valide a reducao e a resposta do LLM no cenario Aurora.
4. Reutilize a mesma estrategia no exemplo real LongBench.
5. Valide a resposta do LLM contra a referencia LongBench.
6. Preencha a conclusao final do notebook.

Nao aumente limites, nao use `FATOS_CRITICOS` dentro da funcao e nao use `answers` para selecionar o contexto LongBench.

In [ ]:
from pathlib import Path

ORCAMENTO_TOKENS = 6_000       # Toda a entrada enviada ao OpenRouter.
ORCAMENTO_CONTEXTO = 5_000     # Reserva espaco para system prompt e pergunta.
MAX_TOKENS_SAIDA = 400         # Limita somente a resposta gerada.
PERGUNTA = 'Qual foi o incidente Aurora, em qual regiao ocorreu, qual versao resolveu, quem e o responsavel e qual o prazo do relatorio?'

FATOS_CRITICOS = [
    'INCIDENTE: INC-7842',
    'REGIAO: sa-east-1',
    'VERSAO DE ROLLBACK: 4.17.3',
    'RESPONSAVEL: Marina Costa',
    'PRAZO DO RELATORIO: 18/09/2026 17:00 BRT',
]

CAMINHO_DOSSIE = Path('dados/dossie_aurora.txt')
assert CAMINHO_DOSSIE.exists(), 'Dossie ausente. Execute gerar_dossie.py.'
contexto_bruto = CAMINHO_DOSSIE.read_text(encoding='utf-8')
documentos = contexto_bruto.split('\n\n--- FIM DO DOCUMENTO ---\n\n')
print(f'Arquivo: {CAMINHO_DOSSIE}')
print(f'Documentos: {len(documentos)}')
print(f'Caracteres: {len(contexto_bruto):,}')

In [ ]:
def estimar_tokens(texto):
    # Estimativa didatica. Tokenizadores reais variam por modelo.
    return max(1, len(texto) // 4)

tokens_brutos = estimar_tokens(contexto_bruto)
print(f'Tokens estimados: {tokens_brutos:,}')
print(f'Orcamento: {ORCAMENTO_TOKENS:,}')
print(f'Excesso: {tokens_brutos / ORCAMENTO_TOKENS:.1f}x')
assert tokens_brutos > ORCAMENTO_TOKENS, 'O cenario deveria exceder o orcamento.'

In [ ]:
class ContextoExcedido(Exception):
    pass

def validar_orcamento(texto, limite=ORCAMENTO_TOKENS):
    total = estimar_tokens(texto)
    if total > limite:
        raise ContextoExcedido(f'{total:,} tokens estimados para um limite de {limite:,}.')
    return total

# Esta falha e intencional: demonstra que concatenar tudo nao e uma solucao.
try:
    validar_orcamento(contexto_bruto)
except ContextoExcedido as erro:
    print('Falha esperada:', erro)

## TODO 1 - Implemente a reducao
Complete apenas a funcao abaixo. Ela recebe a lista de documentos, a pergunta e o orcamento. Voce pode usar selecao por relevancia, deduplicacao, resumo ou uma combinacao. Nao use `FATOS_CRITICOS` para escolher documentos.

In [ ]:
def reduzir_contexto(documentos, pergunta, orcamento):
    # TODO: selecione e/ou resuma documentos relevantes.
    # A saida deve ser texto, ficar abaixo de `orcamento` e preservar os fatos.
    raise NotImplementedError('Implemente sua politica de contexto.')

# Use 5.000, nao 6.000: a requisicao ainda precisa comportar instrucoes e pergunta.
# contexto_reduzido = reduzir_contexto(documentos, PERGUNTA, ORCAMENTO_CONTEXTO)
# print('Tokens apos reducao:', validar_orcamento(contexto_reduzido, ORCAMENTO_CONTEXTO))

In [ ]:
def avaliar_solucao(contexto_reduzido):
    tokens = validar_orcamento(contexto_reduzido, ORCAMENTO_CONTEXTO)
    encontrados = {fato: fato in contexto_reduzido for fato in FATOS_CRITICOS}
    cobertura = sum(encontrados.values()) / len(FATOS_CRITICOS)
    reducao = 1 - tokens / tokens_brutos
    print(f'Tokens finais: {tokens:,}')
    print(f'Reducao: {reducao:.1%}')
    print(f'Cobertura: {cobertura:.0%}')
    for fato, ok in encontrados.items():
        print('[OK]' if ok else '[FALTOU]', fato)
    assert cobertura == 1, 'A reducao perdeu fatos criticos.'
    return {'tokens': tokens, 'reducao': reducao, 'cobertura': cobertura}

# metricas = avaliar_solucao(contexto_reduzido)

## TODO 2 - Consulte o LLM
Descomente as duas ultimas linhas da proxima celula somente depois que `avaliar_solucao` passar. A chamada deve usar exclusivamente `contexto_reduzido`.

In [ ]:
import os
import requests

ENDPOINT = 'https://openrouter.ai/api/v1/chat/completions'
MODELO = 'cohere/north-mini-code:free'  # Confirme a disponibilidade no catalogo.
SYSTEM_PROMPT = 'Responda apenas com fatos sustentados pelo contexto. Nao invente. Preserve identificadores, nomes, datas e horarios exatamente como aparecem.'

def consultar_llm(contexto, pergunta, modelo=MODELO):
    api_key = os.getenv('OPENROUTER_API_KEY')
    assert api_key, 'Defina OPENROUTER_API_KEY no ambiente.'
    mensagens = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'CONTEXTO:\n{contexto}\n\nPERGUNTA:\n{pergunta}'},
    ]
    entrada_completa = '\n'.join(m['content'] for m in mensagens)
    tokens_entrada = validar_orcamento(entrada_completa, ORCAMENTO_TOKENS)
    print(f'Entrada enviada: {tokens_entrada:,}/{ORCAMENTO_TOKENS:,} tokens estimados')
    resposta = requests.post(
        ENDPOINT,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={
            'model': modelo,
            'messages': mensagens,
            'max_tokens': MAX_TOKENS_SAIDA,
            # Impede o OpenRouter de comprimir o contexto silenciosamente.
            'plugins': [{'id': 'context-compression', 'enabled': False}],
        },
        timeout=90,
    )
    if resposta.status_code == 401:
        raise RuntimeError('Chave OpenRouter ausente, invalida ou expirada.')
    if resposta.status_code == 429:
        raise RuntimeError('Rate limit: escolha outro modelo gratuito disponivel.')
    resposta.raise_for_status()
    return resposta.json()['choices'][0]['message']['content']

# Execute somente depois de implementar e validar `contexto_reduzido`.
# resposta_llm = consultar_llm(contexto_reduzido, PERGUNTA)
# print(resposta_llm)

In [ ]:
def avaliar_resposta_llm(resposta):
    texto = resposta.lower()
    criterios = {
        'incidente': 'inc-7842' in texto,
        'regiao': 'sa-east-1' in texto,
        'versao': '4.17.3' in texto,
        'responsavel': 'marina costa' in texto,
        'prazo': all(valor in texto for valor in ['18/09/2026', '17:00', 'brt']),
    }
    for criterio, ok in criterios.items():
        print('[OK]' if ok else '[FALTOU]', criterio)
    assert all(criterios.values()), 'A resposta do LLM perdeu ou alterou fatos criticos.'
    return criterios

# avaliacao_llm = avaliar_resposta_llm(resposta_llm)

## Entrega
- Notebook executado com a falha inicial, reducao e resposta real do LLM.
- Contexto final abaixo de 6.000 tokens estimados e cobertura local de 100% dos fatos.
- Resposta do LLM aprovada por `avaliar_resposta_llm`.
- Chat Python ou WPF aplicando a politica escolhida.
- README explicando estrategia, perdas possiveis, custo, latencia e efeito do cache KV em VRAM.

## TODO 3 - Aplique no LongBench
Esta etapa e obrigatoria. Reutilize `reduzir_contexto` no contexto real abaixo. O campo `answers` serve somente para avaliacao depois da resposta; nao o use para recuperar ou resumir passagens.

In [ ]:
import json
import re

CAMINHO_LONGBENCH = Path('dados/longbench_hotpotqa_exemplo.json')
exemplo_real = json.loads(CAMINHO_LONGBENCH.read_text(encoding='utf-8'))
documentos_longbench = [p for p in re.split(r'(?=Passage \d+:\n)', exemplo_real['context']) if p.strip()]
pergunta_longbench = exemplo_real['input']
print('Fonte:', exemplo_real['source'])
print('Pergunta:', pergunta_longbench)
print('Passagens:', len(documentos_longbench))
print('Tokens estimados:', f"{estimar_tokens(exemplo_real['context']):,}")

# TODO: use a mesma funcao criada no cenario Aurora.
# contexto_longbench = reduzir_contexto(documentos_longbench, pergunta_longbench, ORCAMENTO_CONTEXTO)
# validar_orcamento(contexto_longbench, ORCAMENTO_CONTEXTO)
# resposta_longbench = consultar_llm(contexto_longbench, pergunta_longbench)
# print(resposta_longbench)

In [ ]:
def avaliar_longbench(resposta, respostas_referencia):
    texto = resposta.lower()
    correspondencias = {ref: ref.lower() in texto for ref in respostas_referencia}
    for referencia, ok in correspondencias.items():
        print('[OK]' if ok else '[FALTOU]', referencia)
    assert any(correspondencias.values()), 'A resposta nao corresponde a referencia LongBench.'
    return correspondencias

# avaliacao_longbench = avaliar_longbench(resposta_longbench, exemplo_real['answers'])

## TODO 4 - Conclusao e entrega
Edite esta celula e responda em ate 10 linhas:
- Qual estrategia foi usada nos dois cenarios?
- Quantos tokens havia antes e depois em cada um?
- A mesma estrategia funcionou igualmente no dado sintetico e no real?
- O que poderia ser perdido pela estrategia?
- Como contexto longo afeta custo, latencia e cache KV/VRAM?

Entregue somente este notebook executado, com as duas respostas reais e todas as avaliacoes aprovadas.